In [ ]:
import numpy as np

class DotsAndBoxesStates:
    def __init__(self, h, w):
        self.h = h
        self.w = w

        self.horizontal_edges = np.zeros((h, w-1), dtype=int)
        self.vertical_edges = np.zeros((h-1, w), dtype=int)

        self.player_turn = 1
        self.scores = {1: 0, 2: 0}

        self.boxes = np.zeros((h-1, w-1), dtype=int)
    
    def apply_move(self, type, r, c):
        """
        type: 'h' for horizontal, 'v' for vertical
        r, c: coordinates of the edge
        """

        if type == 'h':
            self.horizontal_edges[r, c] = 1
        else:
            self.vertical_edges[r, c] = 1

        boxes_completed = self._check_and_update_boxes()

        if boxes_completed > 0:
            self.scores[self.player_turn] += boxes_completed
        else:
            self.player_turn = 2 if self.player_turn == 1 else 1
        
        return self

    def _check_and_update_boxes(self):
        new_boxes = 0

        for r in range(self.h - 1):
            for c in range(self.w - 1):
                if self.boxes[r, c] == 0:
                    top = self.horizontal_edges[r, c]
                    bottom = self.horizontal_edges[r + 1, c]
                    left = self.vertical_edges[r, c]
                    right = self.vertical_edges[r, c + 1]

                    if top and bottom and left and right:
                        self.boxes[r, c] = self.player_turn
                        new_boxes += 1
        
        return new_boxes

    def get_legal_moves(self):
        moves = []

        # All horizontal edges
        for r in range(self.h):
            for c in range(self.w - 1):
                if self.horizontal_edges[r, c] == 0:
                    moves.append(('h', r, c))
        
        # All vertical edges
        for r in range(self.h - 1):
            for c in range (self.w):
                if self.vertical_edges[r, c] == 0:
                    moves.append(('v', r, c))
        
        return moves
        
    
    def is_terminal(self):
        return len(self.get_legal_moves()) == 0

    def display(self):
        for r in range(self.h):
            line = ""
            for c in range(self.w - 1):
                line += "● "
                line += "— " if self.horizontal_edges[r, c] == 1 else "  "
            line += "●"
            print(line)

            if r < self.h - 1:
                v_line = ""
                for c in range(self.w):
                    v_line += "| " if self.vertical_edges[r, c] == 1 else "  "
                    if c < self.w - 1:
                        owner = self.boxes[r, c]
                        v_line += f"{owner} " if owner != 0 else "  "
                print(v_line)
    
    def clone(self):
        new_state = DotsAndBoxesStates(self.h, self.w)
        new_state.horizontal_edges = self.horizontal_edges.copy()
        new_state.vertical_edges = self.vertical_edges.copy()
        new_state.boxes = self.boxes.copy()
        new_state.player_turn = self.player_turn
        new_state.scores = self.scores.copy()
        return new_state

    def get_hash(self):
        return (
            tuple(self.horizontal_edges.flatten()),
            tuple(self.vertical_edges.flatten()),
            self.player_turn
        )
    
    def _count_sides(self, state, r, c):
        """
        Counts how many sides of the box at (r, c) are filled.
        """

        if state.horizontals[r][c] == 1: count += 1 
        if state.horizontals[r+1][c] == 1: count += 1
        if state.verticals[r][c] == 1: count += 1
        if state.verticals[r][c+1] == 1: count += 1
        return count

    def _is_safe_move(self, state, move):
        """
        Returns True if the move does NOT complete the 3rd side 
        of any adjacent box.
        """

        edge_type, r, c = move
        if edge_type == 'h':   
            boxes_to_check = []
            if r > 0: boxes_to_check.append((r-1, c))
            if r < state.h - 1: boxes_to_check.append((r, c))
        else:
            boxes_to_check = []
            if c > 0: boxes_to_check.append((r, c-1)) 
            if c < state.w - 1: boxes_to_check.append((r, c))
        
        for br, bc in boxes_to_check:
            if self._count_sides(state, br, bc) == 2:
                return False
        return True

In [18]:
# Example state and moves
state = DotsAndBoxesStates(3, 3)
turn = state.player_turn

print(f"Current turn: Player {turn}")
state.apply_move('h', 0, 0)

turn = state.player_turn
print(f"Current turn: Player {turn}")
state.apply_move('v', 0, 0)

turn = state.player_turn
print(f"Current turn: Player {turn}")
state.apply_move('h', 1, 0)

turn = state.player_turn
print(f"Current turn: Player {turn}")
state.apply_move('v', 0, 1)

turn = state.player_turn
print(f"Current turn: Player {turn}")
state.display()

Current turn: Player 1
Current turn: Player 2
Current turn: Player 1
Current turn: Player 2
Current turn: Player 2
● — ●   ●
| 2 |     
● — ●   ●
          
●   ●   ●


In [ ]:
import math
import random

class MCTSNode:
    def __init__(self, state, parent=None, move=None):
        self.state = state
        self.parent = parent
        self.move = move

        self.children = []
        self.visits = 0
        self.wins = 0.0

        self.untried_actions = state.get_legal_moves()
    
    def is_fully_expanded(self):
        return len(self.untried_actions) == 0
    
    def is_terminal(self):
        return self.state.is_terminal()

    def best_child(self, c_param=1.4):
        # UCB1 formula = (win / visits) + C * sqrt(log(parent_visits) / visits))
        
        choices_weights = [
            (child.wins / child.visits) + c_param * math.sqrt((2 * math.log(self.visits) / child.visits))
            for child in self.children
        ]

        return self.children[choices_weights.index(max(choices_weights))]
    
    def expand(self, node):
        """
        Phase 2: Expand the tree by creating a new child node for one of the untried moves
        """

        move = self.untried_actions.pop()
        next_state = self.state.clone()
        next_state.apply_move(*move)

        state_hash = next_state.get_hash()
        if state_hash in self.transposition_table:
            existing_node = self.transposition_table[state_hash]
            node.children.append(existing_node)
            return existing_node
        else:
            new_node = MCTSNode(next_state, parent=node, move=move)
            self.transposition_table[state_hash] = new_node
            node.children.append(new_node)
            return new_node


In [20]:
# Simulate a random playout from the given node's state

root = MCTSNode(DotsAndBoxesStates(3, 3))
root.expand()
root.expand()
root.expand()

print ("After expanding root node 3 times")
print(f"Check Children: {len(root.children)}")
print (f"Untried Actions: {len(root.untried_actions)}")

print(f"Root state:")
root.state.display()

print(f"First child state:")
root.children[0].state.display()

After expanding root node 3 times
Check Children: 3
Untried Actions: 9
Root state:
●   ●   ●
          
●   ●   ●
          
●   ●   ●
First child state:
●   ●   ●
          
●   ●   ●
        | 
●   ●   ●


In [ ]:
class MCTS:
    def __init__(self, iterations=1000):
        self.iterations = iterations
        self.transposition_table = {} # Store nodes here
    
    def search(self, initial_state):
        root_hash = initial_state.get_hash()
        self.root = MCTSNode(state=initial_state)
        self.transposition_table[root_hash] = self.root

        for _ in range(self.iterations):
            # 1. Selection
            node = self.select(root)

            # 2. Expansion
            if not node.is_terminal():
                node = node.expand()
            
            # 3. Simulation
            result = self.simulate(node.state)

            # 4. Backpropagation
            self.backpropagate(node, result)

        return self.best_action(root)

    def select(self, node):
        """
        Phase 1: Navigate the tree until we find a leaf or unexpanded node.
        """
        
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.best_child()
        return node

    def simulate(self, state):
        """
        Phase 3: Playout. Play randomly until the game ends and return the result. 
        """

        temp_state = state.clone()
        while not temp_state.is_terminal():
            moves = temp_state.get_legal_moves()
            
            # Priority 1: Win
            winning_moves = [m for m in moves if self._would_complete_box(temp_state, m)]
            if winning_moves:
                move = random.choice(winning_moves)
            else:
                # Priority 2: Save moves
                safe_moves = [m for m in moves if self._is_safe_move(temp_state, m)]
                if safe_moves:
                    move = random.choice(safe_moves)
                else:
                    # Priority 3: Random
                    move = random.choice(moves)
            
            temp_state.apply_move(*move)
        
        total_boxes = (temp_state.h - 1) * (temp_state.w - 1)
        p1_score = temp_state.scores[1]
        p2_score = temp_state.scores[2]

        score = (p1_score - p2_score) / total_boxes if total_boxes > 0 else 1e-6
        return score

    def _would_complete_box(self, state, move):
        edge_type, r, c = move

        if edge_type == 'h':
            if r > 0:
                if (state.horizontal_edges[r-1, c] == 1 and 
                    state.vertical_edges[r-1, c] == 1 and 
                    state.vertical_edges[r-1, c+1] == 1):
                    return True
                
            if r < state.h - 1:
                if (state.horizontal_edges[r+1, c] == 1 and 
                    state.vertical_edges[r, c] == 1 and 
                    state.vertical_edges[r, c+1] == 1):
                    return True
        
        else:
            if c > 0:
                if (state.vertical_edges[r, c-1] == 1 and 
                    state.horizontal_edges[r, c-1] == 1 and 
                    state.horizontal_edges[r+1, c-1] == 1):
                    return True
                
            if c < state.w - 1:
                if (state.vertical_edges[r, c+1] == 1 and 
                    state.horizontal_edges[r, c] == 1 and 
                    state.horizontal_edges[r+1, c] == 1):
                    return True
        
        return False
    
    def backpropagate(self, node, result):
        """
        Phase 4: Update the path from child to root.
        """

        while node is not None:
            node.visits += 1

            if node.parent is not None:
                moving_player = node.parent.state.player_turn

                if moving_player == 1:
                    node.wins += result
                else:
                    node.wins -= result
            
            node = node.parent
    
    def best_action(self, root):
        """
        Return the action leading to the best child of the root node.
        """

        best_child_node = max(root.children, key=lambda n: n.visits)
        return best_child_node.move

In [22]:
game_state = DotsAndBoxesStates(5, 5)

mcts_ai = MCTS(iterations=1000)
best_move = mcts_ai.search(game_state)

print(f"The AI suggests moving at: {best_move}")

game_state.apply_move(*best_move)
game_state.display()

The AI suggests moving at: ('h', 3, 1)
●   ●   ●   ●   ●
                  
●   ●   ●   ●   ●
                  
●   ●   ●   ●   ●
                  
●   ● — ●   ●   ●
                  
●   ●   ●   ●   ●


In [27]:
import time
from IPython.display import clear_output

# Main game loop
board_size = 5
game = DotsAndBoxesStates(board_size, board_size)
ai_p1 = MCTS(iterations=600)
ai_p2 = MCTS(iterations=600)

while not game.is_terminal():
    clear_output(wait=True)
    game.display()
    
    current_ai = ai_p1 if game.player_turn == 1 else ai_p2
    print(f"\nAI Player {game.player_turn} is thinking...")
    
    move = current_ai.search(game)
    game.apply_move(*move)
    
    time.sleep(0.1) # Delay for visualization

clear_output(wait=True)
game.display()
print("\nGAME OVER!")
if game.scores[1] > game.scores[2]: print("Winner: Player 1")
elif game.scores[2] > game.scores[1]: print("Winner: Player 2")
else: print("It's a Tie!")

● — ● — ● — ● — ●
| 1 | 1 | 1 | 1 | 
● — ● — ● — ● — ●
| 1 | 1 | 1 | 1 | 
● — ● — ● — ● — ●
| 1 | 1 | 1 | 1 | 
● — ● — ● — ● — ●
| 1 | 1 | 2 | 2 | 
● — ● — ● — ● — ●

GAME OVER!
Winner: Player 1
